In [1]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import dagshub

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

dagshub_token = os.getenv("DAGSHUB_PAT")

if not dagshub_token:
    raise EnvironmentError("DAGSHUB_PAT environment variable is not set")

# DagsHub credentials for MLflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "rajeshxdatascience"
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

# MLflow tracking URI
repo_owner = "rajeshxdatascience"
repo_name = "yt-comment-sentiment-analysis"

mlflow.set_tracking_uri(
    f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow"
)

In [3]:
mlflow.set_experiment("Exp 2 - BoW vs TfIdf")

<Experiment: artifact_location='mlflow-artifacts:/f432bccdf54c492eb9dafafaeb5becb5', creation_time=1786371306657, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786371306657, lifecycle_stage='active', name='Exp 2 - BoW vs TfIdf', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [4]:
df = pd.read_csv(r"C:\Users\rajes\Desktop\yt-comment-sentiment-analysis\data\processed\reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [5]:
df.shape

(36662, 2)

In [7]:
df['category']

0        1
1        1
2       -1
3        0
4        1
        ..
36788    0
36789    1
36790    0
36791    1
36792    0
Name: category, Length: 36662, dtype: int64

In [8]:
# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    # step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest Model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        # Log Random forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log Classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
              for metric, value in metrics.items():
                mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("Confusion Matrix")
        plt.savefig("confusion_matrix.png")
        plt.close()        
        mlflow.log_artifact("confusion_matrix.png") 
                
        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model{vectorizer_name}_{ngram_range}")

# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)] # unigrams, bigrams, trigrams
max_features = 5000

for ngram_range in ngram_ranges:
   # BoW experiments
   run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

   # TF-IDF Experiments
   run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")

2026/08/14 11:35:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 1)_RandomForest at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1/runs/aa816d42cd53482c83848c243bc7f1f4
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1


2026/08/14 11:36:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TF-IDF_(1, 1)_RandomForest at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1/runs/9a7642dcc9544d36882d23791041797a
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1


2026/08/14 11:38:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run TF-IDF_(1, 2)_RandomForest at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1/runs/fbcdf8f7e5ca48669b4dff92b82b8907
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1


2026/08/14 11:42:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 3)_RandomForest at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1/runs/ec29c8f138bb4b1ab1a96e7af17684a9
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1


2026/08/14 11:46:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TF-IDF_(1, 3)_RandomForest at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1/runs/56acaf5292ee40bdaa07dda4b236bab2
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/1
